# AI Inference Music Detector

Find likely matches for a short recording by comparing audio fingerprints. This notebook uses the local SQLite index implemented in `music_recognition.py`.

The included `dataset.csv` contains track metadata, not audio. Add reference recordings and a query recording to run recognition.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from music_recognition import EmptyLibraryError, MusicIndex

PROJECT_ROOT = Path.cwd()
DATASET_PATH = PROJECT_ROOT / "dataset.csv"
REFERENCE_AUDIO_DIR = PROJECT_ROOT / "audio" / "library"
QUERY_AUDIO_PATH = PROJECT_ROOT / "audio" / "query.wav"

## 1. Inspect the metadata

The CSV is useful for browsing track information, but it does not include local audio paths or audio fingerprints. Recognition therefore indexes recordings from `audio/library/` rather than trying to fingerprint the CSV rows.

In [ ]:
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

dataset = pd.read_csv(DATASET_PATH)
print(f"Rows: {len(dataset):,}")
print(f"Columns: {len(dataset.columns)}")
print(f"Has local audio paths: {'audio_path' in dataset.columns}")
display(dataset[["track_name", "artists", "track_genre"]].head())

## 2. Add reference recordings

Install the packages in `requirements.txt` and make FFmpeg available on `PATH`. Put recordings under `audio/library/` in a supported format: WAV, MP3, FLAC, M4A, OGG, or OPUS. Filenames such as `Artist - Track Title.wav` are used to fill in artist and title; other filenames use `Unknown artist` and the filename as the title.

Each recording is fingerprinted and stored in `music_index.sqlite3`. Indexing the same file again updates its existing entry.

In [ ]:
index = MusicIndex()
print(f"Indexed tracks: {index.count()}")
print(f"Index database: {index.database_path}")

## 3. Index the reference library

Run this after adding audio files. Fingerprinting requires the audio backend and FFmpeg; the first run may take a while. If the folder is absent, this cell leaves the metadata-only project untouched.

In [ ]:
if not REFERENCE_AUDIO_DIR.is_dir():
    print(f"Create {REFERENCE_AUDIO_DIR} and add reference recordings to index them.")
else:
    indexed = index.index_directory(REFERENCE_AUDIO_DIR)
    print(f"Indexed or updated {len(indexed)} recording(s).")
    print(f"Total tracks in index: {index.count()}")

## 4. Search a query recording

Place a short query recording at `audio/query.wav`, or change `QUERY_AUDIO_PATH` in the setup cell. The notebook returns the closest indexed fingerprints; the score is a cosine similarity, not a calibrated confidence probability.

In [ ]:
if not QUERY_AUDIO_PATH.is_file():
    print(f"Add a query recording at {QUERY_AUDIO_PATH} to search the library.")
elif index.count() == 0:
    print("The index is empty. Add reference recordings and run the indexing cell first.")
else:
    matches = index.search_file(QUERY_AUDIO_PATH, limit=5)
    display(pd.DataFrame(matches))